In [24]:
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver

In [3]:
# jlist = pd.read_excel('../input/list_of_journals.xlsx')
# jlist = jlist.assign(name='', count=0, url='', num_pages=0, is_completed=False)

In [4]:
# for i in range(len(jlist)):
#     jlist.loc[i, 'name'] = jlist['Journals'][i].split(',')[0]

# jlist.loc[3, 'name'] = 'Quarterly Journal of Economics'
# jlist.loc[12, 'name'] = 'Review of Economics and Statistics'
# jlist.loc[23, 'name'] = 'Journal of Accounting & Economics'
# jlist.loc[38, 'name'] = 'Journal of Money, Credit and Banking'
# jlist.loc[40, 'name'] = 'Journal of Law & Economics'
# jlist.loc[43, 'name'] = 'Journal of Business'
# jlist.loc[52, 'name'] = 'Journal of Economic Dynamics & Control'
# jlist.loc[54, 'name'] = 'Journal of Law, Economics, & Organization'

In [25]:
fp = 'https://apps.webofknowledge.com/WOS_AdvancedSearch_input.do?SID=8Bpl2x46oXogdPbbxt3&product=WOS&search_mode=AdvancedSearch'

In [6]:
jlist = pd.read_csv('save.csv', na_filter=False)

In [9]:
for i in range(len(jlist)):
    if i > 57:
        txt = 'SO=(' + jlist['name'][i] + ')'
        driver = webdriver.Chrome('D:/webdrivers/chromedriver.exe')
        driver.get(fp)
        driver.execute_script("arguments[0].value = arguments[1]", driver.find_element_by_id("value(input1)"), txt)
        driver.find_element_by_id('search-button').click()

        searchResultBtn = 'set_'+ str(i-45) +'_div'
        countStr = driver.find_element_by_id(searchResultBtn).text
        jlist.loc[i, 'count'] = int(countStr.replace(',',''))

        driver.find_element_by_id(searchResultBtn).click()
        jlist.loc[i, 'url'] = driver.current_url
        jlist.loc[i, 'num_pages'] = int(driver.find_element_by_id('pageCount.top').text)

        if jlist['count'][i] < 10000:
            for p in range(jlist['num_pages'][i]):
                if p > 0: 
                    url = jlist['url'][i] + '&action=changePageSize&pageSize=50&page=' + str(p+1)
                    driver = webdriver.Chrome('D:/webdrivers/chromedriver.exe')
                    driver.get(url)

                driver.find_element_by_id('exportTypeName').click()
                driver.find_element_by_id('numberOfRecordsAllOnPage').click()
                driver.find_element_by_id('select2-bib_fields-container').click()
                driver.find_elements_by_class_name('select2-results__option')[3].click()
                driver.find_element_by_id('printButton').click()
                driver.switch_to.window(driver.window_handles[1])
                page = driver.page_source
                soup = BeautifulSoup(page, 'lxml')

                filename_txt = '../output/'+ jlist['name'][i].replace(':','') +'_page_' + str(p+1) + '.txt'
                with open(filename_txt, "w", encoding='utf-8') as file:
                    tbl = soup.findAll("table")
                    for t in range(len(tbl)):
                        if len(tbl[t].findAll('b')) > 1:
                            file.write(tbl[t].get_text())

                driver.quit()
            jlist.loc[i, 'is_completed'] = True
            jlist.to_csv('save.csv', index=False)

In [14]:
large = jlist[jlist['is_completed']==False].reset_index(drop=True)

,Journals,name,count,url,num_pages,is_completed
0,"American Economic Review, American Economic As...",American Economic Review,25244,,505,False
1,"Journal of Political Economy, University of Ch...",Journal of Political Economy,12096,,242,False
2,"Economic Journal, Royal Economic Society",Economic Journal,17752,https://apps.webofknowledge.com/summary.do?pro...,356,False
3,"Energy Policy, Elsevier",Energy Policy,13697,https://apps.webofknowledge.com/summary.do?pro...,274,False
4,"Economics Letters, Elsevier",Economics Letters,11973,https://apps.webofknowledge.com/summary.do?pro...,240,False
5,"Applied Energy, Elsevier",Applied Energy,16591,https://apps.webofknowledge.com/summary.do?pro...,332,False
6,"American Political Science Review, Cambridge U...",American Political Science Review,26612,https://apps.webofknowledge.com/summary.do?pro...,533,False


In [27]:
for i in range(len(large)):
    if i < 6:
        continue
    txt = 'SO=(' + large['name'][i] + ')'
    driver = webdriver.Chrome('D:/webdrivers/chromedriver.exe')
    driver.get(fp)
    driver.execute_script("arguments[0].value = arguments[1]", driver.find_element_by_id("value(input1)"), txt)
    driver.find_element_by_id('search-button').click()

    searchResultBtn = 'set_'+ str(i-5) +'_div'
    countStr = driver.find_element_by_id(searchResultBtn).text
    large.loc[i, 'count'] = int(countStr.replace(',',''))

    driver.find_element_by_id(searchResultBtn).click()
    large.loc[i, 'url'] = driver.current_url
    large.loc[i, 'num_pages'] = int(driver.find_element_by_id('pageCount.top').text)

    for p in range(large['num_pages'][i]):
        if i == 6 and p < 500:
            continue
        if p > 0: 
            url = large['url'][i] + '&action=changePageSize&pageSize=50&page=' + str(p+1)
            driver = webdriver.Chrome('D:/webdrivers/chromedriver.exe')
            driver.get(url)
            
        driver.find_element_by_id('exportTypeName').click()
        driver.find_element_by_id('numberOfRecordsAllOnPage').click()
        driver.find_element_by_id('select2-bib_fields-container').click()
        driver.find_elements_by_class_name('select2-results__option')[3].click()
        driver.find_element_by_id('printButton').click()
        driver.switch_to.window(driver.window_handles[1])
        page = driver.page_source
        soup = BeautifulSoup(page, 'lxml')

        filename_txt = '../output/'+ large['name'][i] +'_page_' + str(p+1) + '.txt'
        with open(filename_txt, "w", encoding='utf-8') as file:
            tbl = soup.findAll("table")
            for t in range(len(tbl)):
                if len(tbl[t].findAll('b')) > 1:
                    file.write(tbl[t].get_text())
        driver.quit()
        
    large.loc[i, 'is_completed'] = True
    jlist.loc[i, 'is_completed'] = True
    large.to_csv('save2.csv', index=False)
    jlist.to_csv('save1.csv', index=False)

In [40]:
out = jlist[['Journals','name','count','num_pages']]
out = out.rename(columns={"name": "Journal Name", "count": "Total Number of Records",'num_pages':'Number of Pages'})
out.to_excel('../WOS Scraping Summary.xlsx',index=False)